# 1d. Strides and tiles

About 25 minutes.

This notebook has no PyTorch payoff. It is here because it is the bridge to
CUDA, where there is no `t[i, j]` and you write the address arithmetic
yourself.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=110)


class _Todo:
    def __repr__(self):
        return "<fill me in>"


TODO = _Todo()


def S(t):
    """Shape as a tuple. Type this constantly."""
    return tuple(t.shape)


def check(label, got, want=None, shape=None, dtype=None, atol=1e-5, rtol=1e-4):
    """Check one answer. Prints what went wrong, then raises."""
    bad = []
    if isinstance(got, _Todo):
        bad.append("not filled in yet")
    else:
        if shape is not None:
            if not torch.is_tensor(got):
                bad.append(f"expected a tensor, got {type(got).__name__}")
            elif S(got) != tuple(shape):
                bad.append(f"shape is {S(got)}, want {tuple(shape)}")
        if dtype is not None and torch.is_tensor(got) and got.dtype != dtype:
            bad.append(f"dtype is {got.dtype}, want {dtype}")
        if want is not None:
            g = got if torch.is_tensor(got) else torch.as_tensor(got)
            w = want if torch.is_tensor(want) else torch.as_tensor(want)
            if S(g) != S(w):
                bad.append(f"shape is {S(g)}, want {S(w)}")
            elif not torch.allclose(g.double(), w.double(), atol=atol, rtol=rtol):
                bad.append(f"values differ, largest gap "
                           f"{(g.double() - w.double()).abs().max().item():.3e}")
    if bad:
        print(f"FAIL  {label}")
        for b in bad:
            print(f"      {b}")
        raise AssertionError(label)
    print(f"ok    {label}")


def same(label, a, b):
    """Two shapes or tuples must match."""
    if isinstance(a, _Todo):
        print(f"FAIL  {label}: not filled in yet")
        raise AssertionError(label)
    if tuple(a) != tuple(b):
        print(f"FAIL  {label}\n      you said {tuple(a)}, it is {tuple(b)}")
        raise AssertionError(label)
    print(f"ok    {label} = {tuple(a)}")

## stride

`t.stride()` says, per axis, how many **elements** to step to advance one
along that axis.

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)

print("shape ", S(t))
print("stride", t.stride())

The address of `t[i, j, k]` is

```text
i*stride[0] + j*stride[1] + k*stride[2]
```

In [ ]:
i, j, k = 1, 2, 3
off = i * t.stride(0) + j * t.stride(1) + k * t.stride(2)

print(f"t[{i},{j},{k}]      = {t[i, j, k].item()}")
print(f"flat[{off}]        = {t.flatten()[off].item()}")

Transposing reorders the strides and touches no data. That is why it is free,
and why `view` then refuses: a view needs strides in descending order.

In [ ]:
tt = t.transpose(0, 2)
print("stride", tt.stride(), " contiguous:", tt.is_contiguous())

## Flat index to row and column

The other direction is *the* CUDA idiom. A kernel gets one flat thread id and
works out which element it owns.

```text
row = idx // N        col = idx % N        for an (K, N) matrix
idx = row * N + col                        the inverse
```

In [ ]:
N = 6
m = torch.arange(4 * N).reshape(4, N)

for idx in (0, 7, 23):
    r, c = idx // N, idx % N
    print(f"idx {idx:>2} -> ({r}, {c}) -> {m[r, c].item():>2}   back: {r*N+c}")

## Drill 1 (easy)

- `off` — the flat offset of `g[1, 1, 2]`, computed **from `g.stride()`**.
  Do not hardcode it.
- `rc` — the `(row, col)` that flat index 23 maps to in a 5x7 matrix.

In [ ]:
g = torch.arange(24).reshape(2, 3, 4)
K_, N_ = 5, 7

off = TODO
rc = TODO

check("off", torch.tensor(off), want=18)
check("off reads the right element", g.flatten()[off], want=g[1, 1, 2])
check("rc", torch.tensor(rc), want=torch.tensor([3, 2]))

## Tiles

No GPU loads a 4096x4096 matrix into fast memory. Every real matmul loops
over tiles and accumulates.

In [ ]:
A = torch.randn(8, 12)
B = torch.randn(12, 6)
out = torch.zeros(8, 6)

for m0 in range(0, 8, 4):
    for n0 in range(0, 6, 2):
        acc = torch.zeros(4, 2)
        for k0 in range(0, 12, 3):              # the reduction loop
            acc += A[m0:m0+4, k0:k0+3] @ B[k0:k0+3, n0:n0+2]
        out[m0:m0+4, n0:n0+2] = acc

print("matches A @ B:", torch.allclose(out, A @ B, atol=1e-5))

Three things to carry forward:

- the `k0` loop is why a kernel needs an accumulator separate from its output
- tile size never changes the answer, only the memory traffic
- except in floating point, where it changes the **summation order**

## Drill 2 (medium)

Compute `W @ x` touching `W` only through `(R, C)` tiles: `R` output rows and
`C` input columns at a time.

`R` is a real tuning knob in the kernel. It must never change the answer.
The last tile in each direction may be short.

In [ ]:
def tiled_matvec(W, x, R, C):
    K, N = W.shape
    out = torch.zeros(K)
    for r0 in range(0, K, R):
        r1 = TODO               # end of this row group, clamped to K
        acc = TODO              # accumulator for this row group
        for c0 in range(0, N, C):
            c1 = TODO
            acc = TODO          # add this tile's contribution
        out[r0:r1] = acc
    return out


W = torch.randn(7, 11)          # divisible by neither R nor C
x = torch.randn(11)

for R, C in [(1, 1), (2, 3), (4, 5), (7, 11), (16, 32)]:
    check(f"R={R:<2} C={C:<2}", tiled_matvec(W, x, R, C), want=W @ x)

Two things that transfer straight to CUDA.

`acc` is a fixed-size local array, which becomes registers. And you write
`out` once, after the column loop, not inside it.

## Next

**1e** — blank dimensions. The most useful notebook in this lesson.